# Where does the experiment/simulation discrepancy come from?

The prefilter rejects background more effectively in simulation than on data. Since
experimental events at this stage are atmospheric muons almost without exception, the two
samples are the same physical class and the comparison is direct.

All scoring here uses the network trained **without** domain adaptation: DA trains the two
domains to look alike, so a DA score cannot test whether they do.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[3]))

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
import pandas as pd

from inference import reader

# The no-DA control is the diagnostic instrument here: domain adaptation trains the two
# domains to look alike, so a DA score cannot be used to test whether they do.
CHECKPOINT_WITHOUT_DA = "da_prefilter_numu_260428_2039"
CHECKPOINT_WITH_DA    = "da_prefilter_numu_260429_0155"

BACKGROUND_MC  = ["e.data_class = 'muatm_2020'"]
H8S3_SIGNOISE  = [reader.H8S3]
EXPERIMENT_OK  = [reader.NOT_EXCLUDED]

TARGET_MC_SUPPRESSION = 1e-3
SCORE_BINS            = 1000

# Hit-multiplicity bands used wherever the comparison has to be made at fixed occupancy.
HIT_BANDS = [(8, 40), (40, 60), (60, 80), (80, 100), (100, 120), (120, 166), (166, 10_000)]
SIGNAL_HIT_BANDS = [0, 1, 2, 3, 4, 6, 8, 12, 20, 40, 100_000]
CUBE_HIT_EDGES   = [8, 20, 30, 40, 50, 60, 70, 80, 100, 120, 166, 100_000]

# A quality cut built from charge alone: no network of any kind takes part in it.
BRIGHT_CHARGE_THRESHOLDS = [3.0, 5.0, 10.0]     # p.e.
BRIGHT_HIT_REQUIREMENTS  = [2, 3, 5]
SIMULATION_PARTS_FOR_CHARGE = 250
EXPERIMENT_EVENTS_PER_PART  = 500_000

# Section 1b works on the bright-hit selection, contrasting a loose cut (where the two
# domains agree) with a tight one (where they do not), at a fixed hit multiplicity.
SHAPE_CHARGE_THRESHOLD = 5.0
SHAPE_HIT_BAND         = (60, 100)
STRING_DIVISOR         = 36

# Section 6 bins events on every feature at once. Each axis is coarse on purpose: the
# number of cells is the product of the axis lengths, and a ratio built on a handful of
# events per cell is noise wearing the costume of a measurement.
FEATURE_AXES = {
    "n_hits":                           [8, 40, 60, 80, 100, 120, 166, np.inf],
    "bright_hits":                      [1, 2, 3, 4, 6, 8, 12, np.inf],
    "time_span_of_bright_hits_ns":      [0, 200, 500, 1000, 2000, 5000, np.inf],
    "vertical_span_of_bright_hits_m":   [0, 30, 60, 100, 150, 200, np.inf],
    "horizontal_span_of_bright_hits_m": [0, 30, 60, 100, 150, np.inf],
    "strings_with_a_bright_hit":        [1, 2, 3, 4, 6, np.inf],
}
SCORE_AXIS = [0, 0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0001]
MINIMUM_EVENTS_PER_FEATURE_CELL = 25

DATA_LABEL = "experiment (exp_full)"
MC_LABEL   = "simulation (muatm)"
DATA_COLOR, MC_COLOR = "#8172b2", "#c44e52"

# h8s3 keeps 2.4% of exp_full, so reading those events' hits means reading the spans
# between them too: measured at ~50 events/s. Charge and timing therefore come from a
# small, explicitly sized sample; everything on full statistics avoids hits entirely.
EVENTS_FOR_HIT_LEVEL = 20_000
HIT_SAMPLE_DIVISOR   = 256

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

## Tools

In [ ]:
def score_histogram(source, checkpoint, cuts, by=None, bins=SCORE_BINS):
    return reader.open(source, checkpoint=checkpoint).histogram(
        "score", bins=bins, span=(0.0, 1.0), where=cuts, by=by)


def observable_histogram(source, checkpoint, cuts, column, span, bins=120):
    """Any per-event column of the prediction table, binned in the database."""
    return reader.open(source, checkpoint=checkpoint).histogram(
        column, bins=bins, span=span, where=cuts)


def score_histogram_by_hit_band(source, checkpoint, cuts, bands):
    """Score histogram within each hit-multiplicity band, and the band occupancies."""
    counts, occupancy = [], []
    for low, high in bands:
        in_band = cuts + [f"p.n_hits >= {low} AND p.n_hits < {high}"]
        binned = score_histogram(source, checkpoint, in_band)["n"].to_numpy(float)
        counts.append(binned)
        occupancy.append(binned.sum())
    occupancy = np.array(occupancy)
    return np.array(counts), occupancy / occupancy.sum()


def reweighted_to_shape(band_counts, target_occupancy):
    """The same events, re-mixed so the hit-multiplicity distribution matches a target.

    Each band keeps its own score distribution; only how much each band contributes
    changes. So whatever survives this is not explained by the multiplicity spectrum.
    """
    per_band = band_counts.sum(axis=1)
    scale = np.divide(target_occupancy, per_band / per_band.sum(),
                      out=np.zeros_like(target_occupancy), where=per_band > 0)
    return (band_counts * scale[:, None]).sum(axis=0)


def _band_expression(column, edges):
    cases = " ".join(f"WHEN {column} < {high} THEN {i}"
                     for i, high in enumerate(edges[1:]))
    return f"CASE {cases} ELSE {len(edges) - 2} END"


def score_cube(source, selector, hit_edges, signal_edges, bins=SCORE_BINS):
    """counts[hit band, signal-hit band, score bin] — one pass over the database.

    No quality cut is applied: h8s3 is itself a selection that behaves differently in the
    two domains, so it cannot be part of a comparison meant to control for that.
    """
    connection = reader.query.connect(reader.open_checkpoint(CHECKPOINT_WITHOUT_DA), source)
    frame = connection.execute(f"""
        SELECT {_band_expression('p.n_hits', hit_edges)}      AS hit_band,
               {_band_expression('p.n_sn_hits', signal_edges)} AS signal_band,
               least(cast(floor(p.score * {bins}) AS INTEGER), {bins - 1}) AS score_bin,
               count(*) AS n
        FROM predictions p
        JOIN cat.events e ON e.id = p.event_fk
        JOIN cat.h5_locations l ON l.event_fk = p.event_fk
        WHERE {" AND ".join(selector)} AND p.n_hits >= {hit_edges[0]}
        GROUP BY 1, 2, 3
    """).df()
    connection.close()
    counts = np.zeros((len(hit_edges) - 1, len(signal_edges) - 1, bins))
    counts[frame.hit_band, frame.signal_band, frame.score_bin] = frame.n
    return counts


def matched_score_distribution(observed_cube, target_cube, common_cells):
    """The experiment re-mixed so that each (hits, signal hits) cell carries the same
    share as in the simulation. Cells outside the common region are dropped from both."""
    observed_share = np.where(common_cells, observed_cube.sum(axis=2), 0.0)
    target_share = np.where(common_cells, target_cube.sum(axis=2), 0.0)
    observed_share /= observed_share.sum()
    target_share /= target_share.sum()
    scale = np.divide(target_share, observed_share, out=np.zeros_like(target_share),
                      where=observed_share > 0)
    return (np.where(common_cells[..., None], observed_cube, 0.0)
            * scale[..., None]).sum(axis=(0, 1))


def _charge_column(group, part_key, n_events):
    """The charge of every hit of the first `n_events` events of one part.

    Read straight from HDF5 and contiguously. The reader's per-event path is the wrong
    tool here: for a selection this sparse it reads 68 hits for every one it keeps,
    measured, because merged runs span the gaps between wanted events.
    """
    boundaries = group[f"raw/ev_starts/{part_key}/data"][:].astype(np.int64)
    n_events = min(n_events or len(boundaries) - 1, len(boundaries) - 1)
    charge = group[f"raw/data/{part_key}/data"][:int(boundaries[n_events]), 0]
    return charge, boundaries[:n_events + 1]


def bright_hit_counts(source, parts, group_name, events_per_part=None,
                      thresholds=BRIGHT_CHARGE_THRESHOLDS):
    """Per event: total hits, and how many of them are brighter than each threshold."""
    collected = []
    with h5py.File(reader.spec.SOURCES[source].h5, "r") as handle:
        group = handle[group_name]
        for part_key in parts:
            charge, boundaries = _charge_column(group, part_key, events_per_part)
            starts = boundaries[:-1].astype(np.intp)
            frame = pd.DataFrame({
                "part_key": part_key,
                "local_idx": np.arange(len(starts)),
                "n_hits": (boundaries[1:] - boundaries[:-1]).astype(np.int32),
            })
            for threshold in thresholds:
                frame[f"hits_above_{threshold:g}_pe"] = np.add.reduceat(
                    (charge > threshold).astype(np.int32), starts)
            collected.append(frame)
    return pd.concat(collected, ignore_index=True)


def scores_by_location(source, parts, data_class=None):
    """Scores keyed by (part, local index).

    `data_class` is not optional for MC: part names repeat across muatm, nuatm and nue2,
    so joining on the part key alone silently pulls in neutrinos and duplicates events.
    """
    connection = reader.query.connect(reader.open_checkpoint(CHECKPOINT_WITHOUT_DA), source)
    listed = ", ".join(f"'{part}'" for part in parts)
    restrict = f"AND e.data_class = '{data_class}'" if data_class else ""
    frame = connection.execute(f"""
        SELECT l.part_key, l.local_idx, p.score
        FROM predictions p
        JOIN cat.events e ON e.id = p.event_fk
        JOIN cat.h5_locations l ON l.event_fk = p.event_fk
        WHERE l.part_key IN ({listed}) {restrict}
    """).df()
    connection.close()
    assert not frame.duplicated(["part_key", "local_idx"]).any(), \
        f"{source}: (part_key, local_idx) is not unique — the join is wrong"
    return frame


def tail_matched_in_hit_count(simulation, experiment, hit_edges, bins=SCORE_BINS,
                              minimum_per_band=200):
    """Tail fractions, with the experiment re-mixed to the simulation's n_hits spectrum."""
    n_bands = len(hit_edges) - 1
    histogram = {}
    for name, frame in (("simulation", simulation), ("experiment", experiment)):
        band = np.digitize(frame.n_hits, hit_edges) - 1
        histogram[name] = np.array([
            np.histogram(frame.score[band == b], bins=bins, range=(0, 1))[0]
            for b in range(n_bands)], dtype=float)

    common = np.logical_and(*[h.sum(axis=1) >= minimum_per_band
                              for h in histogram.values()])
    share = {name: np.where(common, h.sum(axis=1), 0.0) for name, h in histogram.items()}
    share = {name: value / value.sum() for name, value in share.items()}
    scale = np.divide(share["simulation"], share["experiment"],
                      out=np.zeros_like(share["simulation"]), where=share["experiment"] > 0)

    simulated = np.where(common[:, None], histogram["simulation"], 0.0).sum(axis=0)
    matched = (np.where(common[:, None], histogram["experiment"], 0.0)
               * scale[:, None]).sum(axis=0)
    tail = lambda counts: np.cumsum(counts[::-1])[::-1] / counts.sum()
    return tail(simulated), tail(matched), int(common.sum())


def event_shape_observables(source, parts, group_name, threshold=SHAPE_CHARGE_THRESHOLD,
                            events_per_part=None):
    """Per event: what its bright hits look like, and the event's overall time span.

    Restricted to the bright hits on purpose. An observable summed over all hits is
    dominated by whichever domain has more of them, and the two domains differ there;
    the bright hits are the ones the excess grows with.
    """
    collected = []
    with h5py.File(reader.spec.SOURCES[source].h5, "r") as handle:
        group = handle[group_name]
        for part_key in parts:
            boundaries = group[f"raw/ev_starts/{part_key}/data"][:].astype(np.int64)
            n_events = min(events_per_part or len(boundaries) - 1, len(boundaries) - 1)
            last_hit = int(boundaries[n_events])
            hits = group[f"raw/data/{part_key}/data"][:last_hit]
            channels = group[f"raw/channels/{part_key}/data"][:last_hit].astype(np.int64)
            charge, time_ns, depth = hits[:, 0], hits[:, 1], hits[:, 4]

            per_event = (boundaries[1:n_events + 1] - boundaries[:n_events]).astype(np.int64)
            event_of_hit = np.repeat(np.arange(n_events), per_event)
            bright = charge > threshold
            n_bright = np.bincount(event_of_hit[bright], minlength=n_events)
            has_bright = n_bright > 0
            bright_starts = np.concatenate([[0], np.cumsum(n_bright)])[:-1][has_bright]

            frame = pd.DataFrame({"part_key": part_key,
                                  "local_idx": np.arange(n_events),
                                  "n_hits": per_event.astype(np.int32),
                                  "bright_hits": n_bright.astype(np.int32)})
            for name in ("time_span_of_bright_hits_ns", "vertical_span_of_bright_hits_m",
                         "horizontal_span_of_bright_hits_m",
                         "brightest_hit_pe", "charge_in_bright_hits_pe"):
                frame[name] = np.nan
            reduce_over = lambda values: (np.maximum.reduceat(values, bright_starts),
                                          np.minimum.reduceat(values, bright_starts))
            bright_time = time_ns[bright]
            bright_depth = depth[bright]
            high, low = reduce_over(bright_time)
            frame.loc[has_bright, "time_span_of_bright_hits_ns"] = high - low
            high, low = reduce_over(bright_depth)
            frame.loc[has_bright, "vertical_span_of_bright_hits_m"] = high - low
            across, along = (reduce_over(hits[bright, 2]), reduce_over(hits[bright, 3]))
            frame.loc[has_bright, "horizontal_span_of_bright_hits_m"] = np.hypot(
                across[0] - across[1], along[0] - along[1])
            frame.loc[has_bright, "brightest_hit_pe"] = np.maximum.reduceat(
                charge[bright], bright_starts)
            frame.loc[has_bright, "charge_in_bright_hits_pe"] = np.add.reduceat(
                charge[bright], bright_starts)

            keyed = event_of_hit[bright] * 1000 + channels[bright] // STRING_DIVISOR
            frame["strings_with_a_bright_hit"] = np.bincount(
                (np.unique(keyed) // 1000).astype(np.int64),
                minlength=n_events).astype(np.int32)

            starts = boundaries[:n_events].astype(np.intp)
            frame["time_span_of_all_hits_ns"] = (np.maximum.reduceat(time_ns, starts)
                                                 - np.minimum.reduceat(time_ns, starts))
            collected.append(frame)
    return pd.concat(collected, ignore_index=True)


def feature_cube(frame, axes):
    """Counts over every axis at once. Values outside an axis fall into its end bins."""
    position = tuple(np.clip(np.digitize(frame[name].to_numpy(float), edges) - 1,
                             0, len(edges) - 2)
                     for name, edges in axes.items())
    counts = np.zeros(tuple(len(edges) - 1 for edges in axes.values()))
    np.add.at(counts, position, 1)
    return counts


def matched_on_every_feature(simulated, observed, minimum=MINIMUM_EVENTS_PER_FEATURE_CELL):
    """Score distributions with the experiment re-mixed cell by cell onto the simulation.

    The last axis is the score; every earlier axis is a feature being controlled for.
    Cells too thin in either domain are dropped from both, and how much of each sample
    that costs is reported beside the result — a ratio computed on the leftovers would
    look like a controlled comparison without being one.
    """
    occupancy = {name: cube.sum(axis=-1) for name, cube in
                 (("simulation", simulated), ("experiment", observed))}
    common = np.logical_and(*[value >= minimum for value in occupancy.values()])
    share = {name: np.where(common, value, 0.0) for name, value in occupancy.items()}
    share = {name: value / value.sum() for name, value in share.items()}
    scale = np.divide(share["simulation"], share["experiment"],
                      out=np.zeros_like(share["simulation"]),
                      where=share["experiment"] > 0)

    over_features = tuple(range(common.ndim))
    keep = lambda cube: np.where(common[..., None], cube, 0.0)
    return {
        "simulation": keep(simulated).sum(over_features),
        "experiment": keep(observed).sum(over_features),
        "experiment matched": (keep(observed) * scale[..., None]).sum(over_features),
        "common cells": int(common.sum()),
        "cells": int(common.size),
        "kept of simulation": occupancy["simulation"][common].sum()
                              / occupancy["simulation"].sum(),
        "kept of experiment": occupancy["experiment"][common].sum()
                              / occupancy["experiment"].sum(),
    }


def quantiles_from_histogram(centres, counts, probabilities=(0.25, 0.5, 0.75)):
    cumulative = np.cumsum(counts) / counts.sum()
    return np.interp(probabilities, cumulative, centres)


def fraction_above(counts, bin_index):
    counts = np.asarray(counts, float)
    return counts[bin_index:].sum() / counts.sum() if counts.sum() else np.nan


def bin_index_for_mc_suppression(mc_counts, target=TARGET_MC_SUPPRESSION):
    """First score bin whose MC tail fraction is at or below the target."""
    tail = np.cumsum(mc_counts[::-1])[::-1] / mc_counts.sum()
    return int(np.argmax(tail <= target))


def excess_versus_threshold(mc_counts, data_counts):
    mc_tail = np.cumsum(mc_counts[::-1])[::-1] / mc_counts.sum()
    data_tail = np.cumsum(data_counts[::-1])[::-1] / data_counts.sum()
    with np.errstate(divide="ignore", invalid="ignore"):
        return mc_tail, data_tail, np.where(mc_tail > 0, data_tail / mc_tail, np.nan)


def load_events_with_truth(checkpoint, cuts):
    columns = ["event_fk", "score", "n_hits", "n_sn_hits", "n_sn_strings",
               "prime_theta_deg", "prime_energy_gev", "prime_response_muons_n",
               "prime_event_weight"]
    handle = reader.open("mc_merged", checkpoint=checkpoint)
    events = pd.concat([chunk.events[columns]
                        for chunk in handle.stream(where=cuts, with_hits=False)],
                       ignore_index=True)
    return events.rename(columns={"prime_theta_deg": "zenith_deg",
                                  "prime_energy_gev": "primary_energy_gev",
                                  "prime_response_muons_n": "muon_multiplicity",
                                  "prime_event_weight": "event_weight"})


def summarise_events_from_hits(source, checkpoint, cuts, max_events, budget_mb=384):
    """Per-event observables built from raw hits — nothing here involves the prefilter."""
    sampled = cuts + [f"hash(p.event_fk) % {HIT_SAMPLE_DIVISOR} = 0"]
    handle = reader.open(source, checkpoint=checkpoint)
    collected, seen = [], 0
    for chunk in handle.stream(where=sampled, budget_mb=budget_mb):
        by_event = chunk.hits.groupby("event")
        summary = pd.DataFrame({
            "n_hits":            by_event.size(),
            "total_charge_pe":   by_event["q"].sum(),
            "largest_hit_pe":    by_event["q"].max(),
            "median_hit_pe":     by_event["q"].median(),
            "time_span_ns":      by_event["t"].max() - by_event["t"].min(),
            "n_strings":         by_event["string"].nunique(),
            "n_modules":         by_event["channel"].nunique(),
            "signal_hit_share":  by_event["is_sig"].mean(),
        })
        summary["score"] = chunk.events["score"].to_numpy()[summary.index]
        collected.append(summary.reset_index(drop=True))
        seen += len(summary)
        if seen >= max_events:
            break
    return pd.concat(collected, ignore_index=True).iloc[:max_events]


def compare_distributions(data, simulation, columns, log_x=(), n_bins=60):
    figure, axes = plt.subplots(2, (len(columns) + 1) // 2,
                                figsize=(4.4 * ((len(columns) + 1) // 2), 7))
    for axis, column in zip(axes.ravel(), columns):
        both = np.concatenate([data[column], simulation[column]])
        lo, hi = np.percentile(both, [0.2, 99.8])
        edges = (np.logspace(np.log10(max(lo, 1e-3)), np.log10(hi), n_bins)
                 if column in log_x else np.linspace(lo, hi, n_bins))
        for frame, label, color in ((data, DATA_LABEL, DATA_COLOR),
                                    (simulation, MC_LABEL, MC_COLOR)):
            counts, _ = np.histogram(frame[column], bins=edges)
            axis.step(0.5 * (edges[:-1] + edges[1:]), counts / counts.sum(),
                      where="mid", color=color, lw=1.7, label=label)
        if column in log_x:
            axis.set_xscale("log")
        axis.set(xlabel=column, ylabel="fraction of events", yscale="log")
        axis.grid(alpha=0.25)
    axes.ravel()[0].legend(fontsize=8)
    for axis in axes.ravel()[len(columns):]:
        axis.axis("off")
    figure.tight_layout()
    return figure


def median_comparison_table(data, simulation, columns):
    return pd.DataFrame({
        "median in experiment": [data[c].median() for c in columns],
        "median in simulation": [simulation[c].median() for c in columns],
        "experiment / simulation": [data[c].median() / simulation[c].median()
                                    for c in columns],
    }, index=columns)


def reweighting_ceiling(events, variable, edges, bin_index, weights=None):
    """How much a reweighting of one truth variable could possibly raise the MC tail.

    Putting all of the MC's weight into whichever bin of `variable` passes the cut most
    often is the largest tail fraction any reweighting can reach. If the experiment sits
    above that ceiling, no reweighting of this variable explains it — whatever the
    reweighting's physical merit.
    """
    which = np.digitize(events[variable].to_numpy(float), edges) - 1
    inside = (which >= 0) & (which < len(edges) - 1)
    weights = (np.ones(len(events)) if weights is None
               else events[weights].to_numpy(float))
    passes = events["score"].to_numpy(float) >= bin_index / SCORE_BINS

    n_bins = len(edges) - 1
    total = np.bincount(which[inside], weights=weights[inside], minlength=n_bins)
    passing = np.bincount(which[inside & passes], weights=weights[inside & passes],
                          minlength=n_bins)
    with np.errstate(divide="ignore", invalid="ignore"):
        pass_rate = np.where(total > 0, passing / total, np.nan)
    occupancy = total / total.sum()
    populated = total > 0.001 * total.sum()   # ignore bins too empty to reweight into
    return pd.DataFrame({"bin_low": edges[:-1], "bin_high": edges[1:],
                         "share_of_mc": occupancy, "pass_rate": pass_rate,
                         "reweightable": populated})


def save(figure, name):
    figure.savefig(FIGURES / name, dpi=140, bbox_inches="tight")
    print(FIGURES / name)

## The discrepancy, restated

In [ ]:
mc_counts   = score_histogram("mc_merged", CHECKPOINT_WITHOUT_DA,
                              H8S3_SIGNOISE + BACKGROUND_MC)["n"].to_numpy(float)
data_counts = score_histogram("exp_full", CHECKPOINT_WITHOUT_DA,
                              H8S3_SIGNOISE + EXPERIMENT_OK)["n"].to_numpy(float)

working_bin = bin_index_for_mc_suppression(mc_counts)
mc_tail, data_tail, excess = excess_versus_threshold(mc_counts, data_counts)

pd.DataFrame({
    "events passing h8s3": [int(mc_counts.sum()), int(data_counts.sum())],
    "tail fraction at the working point": [mc_tail[working_bin], data_tail[working_bin]],
}, index=[MC_LABEL, DATA_LABEL]).assign(**{
    "score threshold": working_bin / SCORE_BINS,
    "experiment / simulation": [1.0, excess[working_bin]]})

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4.4))
score_axis = np.arange(SCORE_BINS) / SCORE_BINS

axes[0].step(score_axis, mc_tail, where="post", color=MC_COLOR, lw=1.8, label=MC_LABEL)
axes[0].step(score_axis, data_tail, where="post", color=DATA_COLOR, lw=1.8,
             label=DATA_LABEL)
axes[0].axhline(TARGET_MC_SUPPRESSION, color="0.4", lw=0.9, ls=":")
axes[0].set(yscale="log", xlabel="score threshold",
            ylabel="fraction of h8s3 events above threshold",
            title="what fraction survives the cut")
axes[0].legend(fontsize=9)

axes[1].step(score_axis, excess, where="post", color="0.2", lw=1.8)
axes[1].axhline(1.0, color="0.4", lw=0.9, ls=":")
axes[1].axvline(working_bin / SCORE_BINS, color="0.4", lw=0.9, ls=":")
axes[1].set(xlabel="score threshold", ylabel="experiment / simulation",
            title="how much more often the experiment survives", ylim=(0.8, 4))

for axis in axes:
    axis.grid(alpha=0.25)
figure.suptitle("Background rejection: simulation against experiment (network without DA)",
                y=1.02)
figure.tight_layout()
save(figure, "d0_background_rejection.png")

## 1. Is the prefilter implicated at all?

### 1a. Event observables, full statistics

`n_hits` is written by the converter and involves no network at all; the two `n_sn_`
columns come from the sig-noise network.

In [ ]:
EVENT_OBSERVABLES = {
    "n_hits":       (0, 600),    # written by the converter — no network involved
    "n_sn_hits":    (0, 300),    # sig-noise network
    "n_sn_strings": (0, 40),     # sig-noise network
}

SOURCE_OF = {DATA_LABEL: ("exp_full", H8S3_SIGNOISE + EXPERIMENT_OK),
             MC_LABEL:   ("mc_merged", H8S3_SIGNOISE + BACKGROUND_MC)}
population = {label: reader.open(source, checkpoint=CHECKPOINT_WITHOUT_DA)
                           .count(where=cuts)["events"]
              for label, (source, cuts) in SOURCE_OF.items()}

full_statistics = {}
for column, span in EVENT_OBSERVABLES.items():
    for label, (source, cuts) in SOURCE_OF.items():
        binned = observable_histogram(source, CHECKPOINT_WITHOUT_DA, cuts, column, span)
        full_statistics[label, column] = (binned["mid"].to_numpy(),
                                          binned["n"].to_numpy(float))
        # the histogram drops anything outside the span, which would bias a comparison
        assert binned["n"].sum() == population[label], (label, column, span)

rows = []
for column in EVENT_OBSERVABLES:
    quartiles = {label: quantiles_from_histogram(*full_statistics[label, column])
                 for label in (DATA_LABEL, MC_LABEL)}
    rows.append({
        "observable": column,
        "median in experiment": quartiles[DATA_LABEL][1],
        "median in simulation": quartiles[MC_LABEL][1],
        "experiment / simulation": quartiles[DATA_LABEL][1] / quartiles[MC_LABEL][1],
        "upper quartile in experiment": quartiles[DATA_LABEL][2],
        "upper quartile in simulation": quartiles[MC_LABEL][2],
    })
pd.DataFrame(rows).set_index("observable")

In [ ]:
figure, axes = plt.subplots(1, len(EVENT_OBSERVABLES),
                            figsize=(4.6 * len(EVENT_OBSERVABLES), 3.9))
for axis, column in zip(axes, EVENT_OBSERVABLES):
    for label, color in ((DATA_LABEL, DATA_COLOR), (MC_LABEL, MC_COLOR)):
        centres, counts = full_statistics[label, column]
        axis.step(centres, counts / counts.sum(), where="mid", color=color, lw=1.7,
                  label=label)
    axis.set(xlabel=column, ylabel="fraction of events", yscale="log")
    axis.grid(alpha=0.25)
axes[0].legend(fontsize=8)
figure.suptitle("Event observables on full statistics, h8s3 selection", y=1.03)
figure.tight_layout()
save(figure, "d1a_event_observables.png")

### The parts these hit-level sections read

Out-of-training and covered by the probabilities file, so the same events the rest of the
notebook uses.

In [ ]:
training_parts = {f"part_{n}" for n in json.loads(
    (Path(reader.spec.ROOT) /
     "data_manager/datasets/baikal_mc2020_prefilter/prefilter_h5_parts.json").read_text()
)["muatm"]}
with h5py.File(reader.spec.SOURCES["mc_merged"].probs, "r") as handle:
    parts_with_probabilities = set(handle["muatm_2020"]["n_sig_hits_0.8"].keys())
simulation_parts = sorted(parts_with_probabilities - training_parts,
                          key=lambda part: int(part.split("_")[1]))[:SIMULATION_PARTS_FOR_CHARGE]

with h5py.File(reader.spec.SOURCES["exp_full"].h5, "r") as handle:
    experiment_parts = [part for part in sorted(handle["exp_full"]["raw"]["data"].keys())
                        if part not in reader.spec.SOURCES["exp_full"].bad_runs]

pd.DataFrame({"parts read directly from HDF5":
              {MC_LABEL: len(simulation_parts), DATA_LABEL: len(experiment_parts)},
              "events per part":
              {MC_LABEL: "all", DATA_LABEL: f"first {EXPERIMENT_EVENTS_PER_PART:,}"}})

### 1b. What do the bright hits look like?

Compared inside one hit-multiplicity band, so occupancy cannot contribute, and under the
bright-hit selection of 2d — loose, where the two domains agree on the score, against
tight, where they do not.

In [ ]:
shapes = {
    MC_LABEL:   event_shape_observables("mc_merged", simulation_parts, "muatm_2020"),
    DATA_LABEL: event_shape_observables("exp_full", experiment_parts, "exp_full",
                                        events_per_part=EXPERIMENT_EVENTS_PER_PART),
}

SHAPE_OBSERVABLES = ["time_span_of_bright_hits_ns", "vertical_span_of_bright_hits_m",
                     "strings_with_a_bright_hit", "brightest_hit_pe",
                     "charge_in_bright_hits_pe", "time_span_of_all_hits_ns"]

low, high = SHAPE_HIT_BAND
in_band = {label: frame[(frame.n_hits >= low) & (frame.n_hits < high)]
           for label, frame in shapes.items()}

comparison = {}
for regime, required in (("loose cut (2 bright hits)", 2), ("tight cut (5 bright hits)", 5)):
    selected = {label: frame[frame.bright_hits >= required]
                for label, frame in in_band.items()}
    medians = {label: frame[SHAPE_OBSERVABLES].median()
               for label, frame in selected.items()}
    comparison[regime] = pd.DataFrame({
        MC_LABEL: medians[MC_LABEL], DATA_LABEL: medians[DATA_LABEL],
        "experiment / simulation": medians[DATA_LABEL] / medians[MC_LABEL],
        f"events, {MC_LABEL}": len(selected[MC_LABEL]),
        f"events, {DATA_LABEL}": len(selected[DATA_LABEL])})

pd.concat(comparison, axis=1)

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(14, 7.4))
required = 5
selected = {label: frame[frame.bright_hits >= required]
            for label, frame in in_band.items()}

for axis, observable in zip(axes.ravel(), SHAPE_OBSERVABLES):
    values = np.concatenate([frame[observable].dropna() for frame in selected.values()])
    edges = np.linspace(*np.percentile(values, [0.5, 99.5]), 60)
    for label, color in ((MC_LABEL, MC_COLOR), (DATA_LABEL, DATA_COLOR)):
        counts, _ = np.histogram(selected[label][observable].dropna(), bins=edges)
        axis.step(0.5 * (edges[:-1] + edges[1:]), counts / counts.sum(), where="mid",
                  color=color, lw=1.7, label=label)
    axis.set(xlabel=observable, ylabel="fraction of events")
    axis.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
figure.suptitle(f"Bright hits compared at {SHAPE_HIT_BAND[0]}-{SHAPE_HIT_BAND[1]} hits "
                f"and at least {required} hits above {SHAPE_CHARGE_THRESHOLD:g} p.e.",
                y=1.01)
figure.tight_layout()
save(figure, "d1b_bright_hit_shape.png")

In [ ]:
trend = []
for required in (2, 3, 4, 5, 6, 8):
    selected = {label: frame[frame.bright_hits >= required]
                for label, frame in in_band.items()}
    if min(len(frame) for frame in selected.values()) < 3_000:
        continue
    medians = {label: frame["time_span_of_bright_hits_ns"].median()
               for label, frame in selected.items()}
    trend.append({"bright hits required": required,
                  f"{MC_LABEL}, ns": medians[MC_LABEL],
                  f"{DATA_LABEL}, ns": medians[DATA_LABEL],
                  "experiment / simulation": medians[DATA_LABEL] / medians[MC_LABEL],
                  f"events, {DATA_LABEL}": len(selected[DATA_LABEL])})
pd.DataFrame(trend).set_index("bright hits required")

### 1c. At the same number of hits, are the two domains the same events?

h8s3 counts hits the sig-noise network calls signal. If that count behaves differently in
the two domains, h8s3 is not one selection but two, and every comparison made under it is
between differently chosen events.

In [ ]:
def hit_composition(source, base_cuts, bands=HIT_BANDS):
    """Mean hit composition per band, straight from the prediction table."""
    connection = reader.query.connect(reader.open_checkpoint(CHECKPOINT_WITHOUT_DA), source)
    conditions = " AND ".join(base_cuts) if base_cuts else "TRUE"
    frame = connection.execute(f"""
        SELECT p.n_hits, p.n_sn_hits, p.n_sn_strings, p.n_gt_sig_hits, p.n_gt_sig_strings
        FROM predictions p
        JOIN cat.events e ON e.id = p.event_fk
        JOIN cat.h5_locations l ON l.event_fk = p.event_fk
        WHERE {conditions} AND p.n_hits >= {bands[0][0]} AND p.n_hits < {bands[-1][1]}
    """).df()
    connection.close()

    edges = [low for low, _ in bands] + [bands[-1][1]]
    frame["passes_h8s3"] = (frame.n_sn_hits >= 8) & (frame.n_sn_strings >= 3)
    frame["band"] = pd.cut(frame.n_hits, edges, right=False,
                           labels=[f"{low}-{high}" if high < 10_000 else f"above {low}"
                                   for low, high in bands])
    grouped = frame.groupby("band", observed=True)
    return pd.DataFrame({
        "events": grouped.size(),
        "mean hits": grouped.n_hits.mean(),
        "mean signal hits (network)": grouped.n_sn_hits.mean(),
        "signal share of hits": grouped.n_sn_hits.mean() / grouped.n_hits.mean(),
        "passes h8s3": grouped.passes_h8s3.mean(),
    })


SIMULATION_SELECTOR = ["e.data_class = 'muatm_2020'"]
EXPERIMENT_SELECTOR = ["l.part_key NOT IN "
                       "('part_s2020_c02_r0020', 'part_s2020_c02_r0249')"]

composition = pd.concat(
    {MC_LABEL:   hit_composition("mc_merged", SIMULATION_SELECTOR),
     DATA_LABEL: hit_composition("exp_full", EXPERIMENT_SELECTOR)},
    axis=1)
composition

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
positions = np.arange(len(composition))

for column, axis, title in (("signal share of hits", axes[0],
                             "how much of an event the sig-noise network calls signal"),
                            ("passes h8s3", axes[1],
                             "how often h8s3 is satisfied")):
    for label, color, offset in ((MC_LABEL, MC_COLOR, -0.2),
                                 (DATA_LABEL, DATA_COLOR, 0.2)):
        axis.bar(positions + offset, composition[label][column], width=0.4,
                 color=color, alpha=0.85, label=label)
    axis.set(xticks=positions, ylabel=column, title=title)
    axis.set_xticklabels(composition.index, rotation=25, ha="right")
    axis.set_xlabel("n_hits band")
    axis.grid(alpha=0.25, axis="y")
axes[0].legend(fontsize=9)
figure.suptitle("At the same number of hits, are the two domains the same events?", y=1.03)
figure.tight_layout()
save(figure, "d1c_hit_composition.png")

### 1d. How far can the sig-noise network be trusted?

Only the simulation has truth to check it against. Whatever the network does on data
cannot be verified this way — which is the limit of every statement in 1c.

In [ ]:
connection = reader.query.connect(
    reader.open_checkpoint(CHECKPOINT_WITHOUT_DA), "mc_merged")
network_against_truth = connection.execute(f"""
    SELECT CASE {" ".join(f"WHEN p.n_hits < {high} THEN '{low}-{high}'"
                          for low, high in HIT_BANDS[:-1])}
                ELSE 'above {HIT_BANDS[-1][0]}' END      AS band,
           avg(p.n_gt_sig_hits)                          AS signal_hits_truth,
           avg(p.n_sn_hits)                              AS signal_hits_network,
           avg(p.n_sn_hits) / avg(p.n_gt_sig_hits)       AS network_over_truth,
           avg(CASE WHEN p.n_gt_sig_hits >= 8 AND p.n_gt_sig_strings >= 3
                    THEN 1.0 ELSE 0 END)                 AS passes_h8s3_by_truth,
           avg(CASE WHEN p.n_sn_hits >= 8 AND p.n_sn_strings >= 3
                    THEN 1.0 ELSE 0 END)                 AS passes_h8s3_by_network
    FROM predictions p JOIN cat.events e ON e.id = p.event_fk
    WHERE e.data_class = 'muatm_2020'
      AND p.n_hits >= {HIT_BANDS[0][0]} AND p.n_hits < {HIT_BANDS[-1][1]}
    GROUP BY 1
""").df()
connection.close()

network_against_truth.set_index("band").reindex(
    [f"{low}-{high}" if high < 10_000 else f"above {low}" for low, high in HIT_BANDS])

## 2. Does matching the hit multiplicity remove the discrepancy?

Two confounds at once. The experiment has a different `n_hits` spectrum, and comparing
normalised shapes lets that difference masquerade as a scoring difference — so each band
of `n_hits` is re-mixed to the simulation's occupancy, keeping its own score distribution.
And h8s3 is applied through the sig-noise network, which has a domain behaviour of its
own, so the whole comparison is repeated under a selection no network takes part in.

In [ ]:
band_counts, band_occupancy = {}, {}
for selection_name, extra in (("h8s3 (sig-noise network)", H8S3_SIGNOISE),
                              ("n_hits only (no network)", [])):
    for label, (source, base) in SOURCE_OF.items():
        cuts = (extra + EXPERIMENT_OK if source == "exp_full"
                else extra + BACKGROUND_MC)
        counts, occupancy = score_histogram_by_hit_band(
            source, CHECKPOINT_WITHOUT_DA, cuts, HIT_BANDS)
        band_counts[selection_name, label] = counts
        band_occupancy[selection_name, label] = occupancy

pd.DataFrame({f"{selection} — {label}": band_occupancy[selection, label]
              for selection in ("h8s3 (sig-noise network)", "n_hits only (no network)")
              for label in (MC_LABEL, DATA_LABEL)},
             index=[f"{low}-{high}" if high < 10_000 else f"above {low}"
                    for low, high in HIT_BANDS]).rename_axis("n_hits band")

In [ ]:
matched = {}
for selection in ("h8s3 (sig-noise network)", "n_hits only (no network)"):
    simulated = band_counts[selection, MC_LABEL].sum(axis=0)
    observed_as_is = band_counts[selection, DATA_LABEL].sum(axis=0)
    observed_matched = reweighted_to_shape(band_counts[selection, DATA_LABEL],
                                           band_occupancy[selection, MC_LABEL])
    bin_at_target = bin_index_for_mc_suppression(simulated)
    matched[selection] = {
        "score threshold": bin_at_target / SCORE_BINS,
        "experiment / simulation, as observed":
            excess_versus_threshold(simulated, observed_as_is)[2][bin_at_target],
        "experiment / simulation, hit multiplicity matched":
            excess_versus_threshold(simulated, observed_matched)[2][bin_at_target],
        "_curves": (excess_versus_threshold(simulated, observed_as_is)[2],
                    excess_versus_threshold(simulated, observed_matched)[2]),
    }

pd.DataFrame({name: {k: v for k, v in values.items() if not k.startswith("_")}
              for name, values in matched.items()}).T

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), sharey=True)
for axis, (selection, values) in zip(axes, matched.items()):
    as_observed, multiplicity_matched = values["_curves"]
    axis.step(score_axis, as_observed, where="post", color="0.25", lw=1.8,
              label="as observed")
    axis.step(score_axis, multiplicity_matched, where="post", color="#4c72b0", lw=1.8,
              ls="--", label="hit multiplicity matched to simulation")
    axis.axhline(1.0, color="0.4", lw=0.9, ls=":")
    axis.axvline(values["score threshold"], color="0.4", lw=0.9, ls=":")
    axis.set(xlabel="score threshold", title=selection, ylim=(0.8, 4))
    axis.grid(alpha=0.25)
axes[0].set_ylabel("experiment / simulation")
axes[0].legend(fontsize=9)
figure.suptitle("Does matching the hit multiplicity remove the discrepancy?", y=1.02)
figure.tight_layout()
save(figure, "d2_hit_multiplicity_matched.png")

### 2b. Which events carry the excess?

The left panel says which hit multiplicities the excess is made of; the right says by how
much the experiment out-passes the simulation *within* each band, where the multiplicity
spectrum cannot contribute.

In [ ]:
rows = []
for (low, high), simulated, observed in zip(HIT_BANDS,
                                            band_counts["h8s3 (sig-noise network)", MC_LABEL],
                                            band_counts["h8s3 (sig-noise network)", DATA_LABEL]):
    row = {"n_hits band": f"{low}-{high}" if high < 10_000 else f"above {low}"}
    for label, counts, total in ((MC_LABEL, simulated, population[MC_LABEL]),
                                 (DATA_LABEL, observed, population[DATA_LABEL])):
        row[f"{label}: share of all events"] = counts.sum() / total
        row[f"{label}: share above the working point"] = counts[working_bin:].sum() / total
        row[f"{label}: pass rate within the band"] = (counts[working_bin:].sum()
                                                      / counts.sum() if counts.sum()
                                                      else np.nan)
    row["contribution to the excess"] = (row[f"{DATA_LABEL}: share above the working point"]
                                         - row[f"{MC_LABEL}: share above the working point"])
    # Undefined where the simulation puts nothing above the working point: such a band is
    # not "infinitely worse", it is one the simulation cannot speak about.
    simulated_rate = row[f"{MC_LABEL}: pass rate within the band"]
    row["pass rate, experiment / simulation"] = (
        row[f"{DATA_LABEL}: pass rate within the band"] / simulated_rate
        if simulated_rate else np.nan)
    rows.append(row)

attribution = pd.DataFrame(rows).set_index("n_hits band")
attribution["share of the excess"] = (attribution["contribution to the excess"]
                                      / attribution["contribution to the excess"].sum())
attribution[[f"{MC_LABEL}: share of all events", f"{DATA_LABEL}: share of all events",
             f"{MC_LABEL}: pass rate within the band",
             f"{DATA_LABEL}: pass rate within the band",
             "pass rate, experiment / simulation", "share of the excess"]]

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
positions = np.arange(len(attribution))

axes[0].bar(positions, attribution["share of the excess"], color=DATA_COLOR, alpha=0.85)
axes[0].set(xticks=positions, ylabel="share of the total excess",
            title="where the excess actually sits")
axes[0].set_xticklabels(attribution.index, rotation=25, ha="right")

axes[1].bar(positions, attribution["pass rate, experiment / simulation"],
            color="0.45", alpha=0.85)
axes[1].axhline(1.0, color="0.2", lw=1.0, ls=":")
axes[1].set(xticks=positions, ylabel="experiment / simulation",
            title="pass rate at fixed hit multiplicity")
axes[1].set_xticklabels(attribution.index, rotation=25, ha="right")

for axis in axes:
    axis.grid(alpha=0.25, axis="y")
figure.tight_layout()
save(figure, "d2b_excess_by_hit_multiplicity.png")

### 2c. Same number of hits *and* the same number of signal hits

Sections 1c and 2 leave two ways for the comparison to be unfair: the experiment has a
different hit spectrum, and at any given hit count its events carry far fewer signal hits.
Matching on both at once removes them together. What remains cannot be attributed to how
many hits an event has, nor to how many of them the sig-noise network accepts.

In [ ]:
MINIMUM_EVENTS_PER_CELL = 200

cubes = {MC_LABEL:   score_cube("mc_merged", SIMULATION_SELECTOR,
                                CUBE_HIT_EDGES, SIGNAL_HIT_BANDS),
         DATA_LABEL: score_cube("exp_full", EXPERIMENT_SELECTOR,
                                CUBE_HIT_EDGES, SIGNAL_HIT_BANDS)}
cell_counts = {label: cube.sum(axis=2) for label, cube in cubes.items()}
common_cells = np.logical_and(*[counts >= MINIMUM_EVENTS_PER_CELL
                                for counts in cell_counts.values()])

pd.DataFrame({
    "cells populated in both": [int(common_cells.sum())],
    "cells in total": [int(common_cells.size)],
    f"share of {MC_LABEL} inside them":
        [cell_counts[MC_LABEL][common_cells].sum() / cell_counts[MC_LABEL].sum()],
    f"share of {DATA_LABEL} inside them":
        [cell_counts[DATA_LABEL][common_cells].sum() / cell_counts[DATA_LABEL].sum()],
}, index=["overlap of the two samples"]).T

In [ ]:
inside = lambda cube: np.where(common_cells[..., None], cube, 0.0).sum(axis=(0, 1))
simulated_scores = inside(cubes[MC_LABEL])
observed_scores = inside(cubes[DATA_LABEL])
matched_scores = matched_score_distribution(cubes[DATA_LABEL], cubes[MC_LABEL],
                                            common_cells)

def tail_fraction(counts):
    return np.cumsum(counts[::-1])[::-1] / counts.sum()

simulated_tail = tail_fraction(simulated_scores)
observed_tail = tail_fraction(observed_scores)
matched_tail = tail_fraction(matched_scores)
matched_working_bin = int(np.argmax(simulated_tail <= TARGET_MC_SUPPRESSION))

pd.DataFrame({
    "score threshold": [thr / SCORE_BINS for thr in
                        (0.1 * SCORE_BINS, 0.5 * SCORE_BINS, matched_working_bin,
                         0.9 * SCORE_BINS, 0.95 * SCORE_BINS)],
    "simulation": [simulated_tail[int(i)] for i in
                   (0.1 * SCORE_BINS, 0.5 * SCORE_BINS, matched_working_bin,
                    0.9 * SCORE_BINS, 0.95 * SCORE_BINS)],
    "experiment, unmatched": [observed_tail[int(i)] for i in
                              (0.1 * SCORE_BINS, 0.5 * SCORE_BINS, matched_working_bin,
                               0.9 * SCORE_BINS, 0.95 * SCORE_BINS)],
    "experiment, matched": [matched_tail[int(i)] for i in
                            (0.1 * SCORE_BINS, 0.5 * SCORE_BINS, matched_working_bin,
                             0.9 * SCORE_BINS, 0.95 * SCORE_BINS)],
}).assign(**{"matched / simulation":
             lambda f: f["experiment, matched"] / f["simulation"]}).set_index(
                 "score threshold")

In [ ]:
thresholds = np.arange(SCORE_BINS) / SCORE_BINS
figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

axes[0].step(thresholds, simulated_tail, where="post", color=MC_COLOR, lw=1.8,
             label=MC_LABEL)
axes[0].step(thresholds, observed_tail, where="post", color=DATA_COLOR, lw=1.4, ls=":",
             label=f"{DATA_LABEL}, unmatched")
axes[0].step(thresholds, matched_tail, where="post", color=DATA_COLOR, lw=1.8,
             label=f"{DATA_LABEL}, matched in (n_hits, n_sn_hits)")
axes[0].set(yscale="log", xlabel="score threshold",
            ylabel="fraction above threshold", title="what survives the cut")
axes[0].legend(fontsize=8)

with np.errstate(divide="ignore", invalid="ignore"):
    axes[1].step(thresholds, matched_tail / simulated_tail, where="post", color="0.2",
                 lw=1.8, label="matched")
    axes[1].step(thresholds, observed_tail / simulated_tail, where="post", color="0.55",
                 lw=1.3, ls=":", label="unmatched")
axes[1].axhline(1.0, color="0.4", lw=0.9, ls=":")
axes[1].axvline(matched_working_bin / SCORE_BINS, color="0.4", lw=0.9, ls=":")
axes[1].set(xlabel="score threshold", ylabel="experiment / simulation",
            title="excess at equal hit count and equal signal-hit count", ylim=(0, 3.5))
axes[1].legend(fontsize=8)

for axis in axes:
    axis.grid(alpha=0.25)
figure.suptitle("Same number of hits, same number of signal hits — does the excess remain?",
                y=1.02)
figure.tight_layout()
save(figure, "d2c_matched_hits_and_signal.png")

### 2d. A quality cut that uses no network at all

h8s3 counts hits a network calls signal, and 1c showed that count means different things
in the two domains. Charge does not: bright hits are bright in both. The cut is therefore
"the event contains at least N hits brighter than Q", with nothing learned involved.

It is not calibration-free, though, so the charge spectra are compared first — a threshold
is only a common cut where the two agree.

In [ ]:
CHARGE_SAMPLE_PARTS = 6

charge_samples = {}
for label, source, group_name, parts in (
        (MC_LABEL, "mc_merged", "muatm_2020", None),
        (DATA_LABEL, "exp_full", "exp_full", None)):
    with h5py.File(reader.spec.SOURCES[source].h5, "r") as handle:
        group = handle[group_name]
        available = sorted(group["raw"]["data"].keys())
        if source == "exp_full":
            available = [p for p in available
                         if p not in reader.spec.SOURCES["exp_full"].bad_runs]
        charge_samples[label] = np.concatenate([
            group[f"raw/data/{part}/data"][:400_000, 0]
            for part in available[:CHARGE_SAMPLE_PARTS]])

pd.DataFrame({
    label: {"hits sampled": len(charge), "median charge, p.e.": np.median(charge),
            "90th percentile": np.percentile(charge, 90),
            "99th percentile": np.percentile(charge, 99),
            **{f"share above {t:g} p.e.": (charge > t).mean()
               for t in BRIGHT_CHARGE_THRESHOLDS}}
    for label, charge in charge_samples.items()})

In [ ]:
brightness = {}
for label, source, group_name, parts, cap, data_class in (
        (MC_LABEL, "mc_merged", "muatm_2020", simulation_parts, None, "muatm_2020"),
        (DATA_LABEL, "exp_full", "exp_full", experiment_parts, EXPERIMENT_EVENTS_PER_PART, None)):
    counts = bright_hit_counts(source, parts, group_name, cap)
    brightness[label] = counts.merge(scores_by_location(source, parts, data_class),
                                     on=["part_key", "local_idx"])

pd.DataFrame({label: {"events": len(frame), "parts": frame.part_key.nunique(),
                      "events per part": len(frame) / frame.part_key.nunique()}
              for label, frame in brightness.items()})

In [ ]:
rows = []
for threshold in BRIGHT_CHARGE_THRESHOLDS:
    column = f"hits_above_{threshold:g}_pe"
    for required in BRIGHT_HIT_REQUIREMENTS:
        selected = {label: frame[frame[column] >= required]
                    for label, frame in brightness.items()}
        if min(len(frame) for frame in selected.values()) < 50_000:
            continue
        simulated_tail, matched_tail, bands = tail_matched_in_hit_count(
            selected[MC_LABEL], selected[DATA_LABEL], CUBE_HIT_EDGES)
        at = int(np.argmax(simulated_tail <= TARGET_MC_SUPPRESSION))
        if simulated_tail[at] > TARGET_MC_SUPPRESSION:
            continue
        rows.append({
            "hit brighter than, p.e.": threshold,
            "at least this many": required,
            f"kept of {MC_LABEL}": len(selected[MC_LABEL]) / len(brightness[MC_LABEL]),
            f"kept of {DATA_LABEL}": len(selected[DATA_LABEL]) / len(brightness[DATA_LABEL]),
            "score threshold": at / SCORE_BINS,
            f"{MC_LABEL} tail": simulated_tail[at],
            f"{DATA_LABEL} tail, matched in n_hits": matched_tail[at],
            "experiment / simulation": matched_tail[at] / simulated_tail[at],
        })

bright_excess = pd.DataFrame(rows)
bright_excess

In [ ]:
figure, axis = plt.subplots(figsize=(7.2, 4.4))
for threshold, marker in zip(BRIGHT_CHARGE_THRESHOLDS, ("o", "s", "^")):
    subset = bright_excess[bright_excess["hit brighter than, p.e."] == threshold]
    axis.plot(subset["at least this many"], subset["experiment / simulation"],
              marker=marker, lw=1.8, label=f"hits brighter than {threshold:g} p.e.")
axis.axhline(1.0, color="0.4", lw=0.9, ls=":")
axis.set(xlabel="how many bright hits the event must have",
         ylabel="experiment / simulation at suppression $10^{-3}$",
         title="A quality cut made of charge alone",
         xticks=BRIGHT_HIT_REQUIREMENTS)
axis.grid(alpha=0.25)
axis.legend(fontsize=9)
save(figure, "d2d_bright_hit_quality_cut.png")

### 2e. Every feature at once

One histogram over the score and every event feature measured here, for each domain, and
their ratio bin by bin. The table then re-mixes the experiment cell by cell onto the
simulation, which controls for all six features simultaneously rather than one at a time.

In [ ]:
cube_axes = {**FEATURE_AXES, "score": SCORE_AXIS}
with_bright_hits = {label: frame.dropna(subset=list(FEATURE_AXES))
                    for label, frame in shapes.items()}
scored = {label: frame.merge(
              scores_by_location(source, parts, data_class),
              on=["part_key", "local_idx"])
          for (label, source, parts, data_class), frame in zip(
              ((MC_LABEL, "mc_merged", simulation_parts, "muatm_2020"),
               (DATA_LABEL, "exp_full", experiment_parts, None)),
              with_bright_hits.values())}

cubes = {label: feature_cube(frame, cube_axes) for label, frame in scored.items()}
result = matched_on_every_feature(cubes[MC_LABEL], cubes[DATA_LABEL])

pd.DataFrame({"events with at least one bright hit":
                  {label: len(frame) for label, frame in scored.items()},
              "cells populated in both":
                  {MC_LABEL: result["common cells"], DATA_LABEL: result["cells"]},
              "share of the sample inside those cells":
                  {MC_LABEL: result["kept of simulation"],
                   DATA_LABEL: result["kept of experiment"]}}).T

In [ ]:
above = lambda counts: np.cumsum(counts[::-1])[::-1] / counts.sum()
cumulative = pd.DataFrame({
    "score threshold": SCORE_AXIS[:-1],
    MC_LABEL: above(result["simulation"]),
    DATA_LABEL: above(result["experiment"]),
    f"{DATA_LABEL}, matched on every feature": above(result["experiment matched"]),
})
cumulative["excess, unmatched"] = cumulative[DATA_LABEL] / cumulative[MC_LABEL]
cumulative["excess, matched"] = (
    cumulative[f"{DATA_LABEL}, matched on every feature"] / cumulative[MC_LABEL])
cumulative["experiment events above"] = (above(result["experiment"])
                                         * result["experiment"].sum()).round()
cumulative.set_index("score threshold")

In [ ]:
features = list(FEATURE_AXES)
figure, axes = plt.subplots(2, 3, figsize=(16.5, 8.4))
score_axis_index = len(features)

for axis, feature in zip(axes.ravel(), features):
    keep = (features.index(feature), score_axis_index)
    collapse = tuple(i for i in range(score_axis_index + 1) if i not in keep)
    simulated = cubes[MC_LABEL].sum(collapse)
    observed = cubes[DATA_LABEL].sum(collapse)
    # A ratio built on a handful of events is noise; leave those cells blank.
    thin = (simulated < MINIMUM_EVENTS_PER_FEATURE_CELL) | (observed < MINIMUM_EVENTS_PER_FEATURE_CELL)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(simulated > 0,
                         (observed / observed.sum()) / (simulated / simulated.sum()),
                         np.nan)
    ratio[thin] = np.nan

    image = axis.imshow(np.ma.masked_invalid(ratio.T), origin="lower", aspect="auto",
                        cmap="RdBu_r", norm=LogNorm(vmin=1 / 3, vmax=3))
    image.cmap.set_bad("0.85")
    edges = FEATURE_AXES[feature]
    axis.set_xticks(range(len(edges) - 1),
                    [f"{value:g}" for value in edges[:-1]], rotation=45, fontsize=7)
    axis.set_yticks(range(len(SCORE_AXIS) - 1),
                    [f"{value:g}" for value in SCORE_AXIS[:-1]], fontsize=7)
    axis.set(xlabel=f"{feature} — lower edge of bin", ylabel="score — lower edge of bin")
    figure.colorbar(image, ax=axis, label="experiment / simulation")

figure.suptitle("Where the excess lives. Grey: fewer than "
                f"{MINIMUM_EVENTS_PER_FEATURE_CELL} events in one of the two samples",
                y=1.01)
figure.tight_layout()
save(figure, "d6_feature_cube_projections.png")

## 3. Is it one number, or does it follow the state of the detector?

Simulation is a single static detector configuration; the experiment is 27 runs.

In [ ]:
per_run = score_histogram("exp_full", CHECKPOINT_WITHOUT_DA,
                          H8S3_SIGNOISE + EXPERIMENT_OK, by=["cluster", "run"])
run_rows = []
for (cluster, run), frame in per_run.groupby(["cluster", "run"]):
    counts = frame.sort_values("bin")["n"].to_numpy(float)
    if counts.sum() < 5_000:
        continue
    run_rows.append({"cluster": cluster, "run": run,
                     "events passing h8s3": int(counts.sum()),
                     "tail fraction at the working point":
                         fraction_above(counts, working_bin)})
runs = pd.DataFrame(run_rows)

# query.connect opens the prediction database with the catalog attached as `cat`.
conditions = reader.query.connect(reader.open_checkpoint(CHECKPOINT_WITHOUT_DA),
                                  "exp_full")
run_conditions = conditions.execute("""
    SELECT e.cluster, e.run,
           avg(p.n_hits)                          AS mean_hits_per_event,
           avg(p.n_sn_hits)                       AS mean_signal_hits_per_event,
           avg(p.n_sn_hits) / avg(p.n_hits)       AS signal_share_of_hits,
           avg(p.n_sn_strings)                    AS mean_signal_strings_per_event
    FROM predictions p
    JOIN cat.events e ON e.id = p.event_fk
    WHERE p.n_sn_hits >= 8 AND p.n_sn_strings >= 3
    GROUP BY 1, 2
""").df()
conditions.close()

runs = runs.merge(run_conditions, on=["cluster", "run"])
runs["experiment / simulation"] = (runs["tail fraction at the working point"]
                                   / mc_tail[working_bin])
runs.sort_values("experiment / simulation").reset_index(drop=True)

In [ ]:
drivers = ["mean_hits_per_event", "signal_share_of_hits",
           "mean_signal_strings_per_event"]
figure, axes = plt.subplots(1, len(drivers) + 1,
                            figsize=(4.2 * (len(drivers) + 1), 3.9))

axes[0].hist(runs["experiment / simulation"], bins=14, color=DATA_COLOR, alpha=0.85)
axes[0].axvline(1.0, color="0.3", lw=1.2, ls=":")
axes[0].set(xlabel="experiment / simulation", ylabel="runs",
            title="every run, one entry")

for axis, driver in zip(axes[1:], drivers):
    axis.scatter(runs[driver], runs["experiment / simulation"],
                 c=runs["cluster"], cmap="viridis", s=34)
    correlation = runs[driver].corr(runs["experiment / simulation"])
    axis.set(xlabel=driver, ylabel="experiment / simulation",
             title=f"Pearson r = {correlation:+.2f}")

for axis in axes:
    axis.grid(alpha=0.25)
figure.suptitle("Run-to-run variation of the discrepancy, and what it tracks", y=1.04)
figure.tight_layout()
save(figure, "d3_run_conditions.png")

## 4. Could a different background flux explain it?

Reweighting a truth variable cannot push the simulated tail past the tail of its own
best-performing bin. If the experiment lies above that ceiling, no reweighting of that
variable explains the discrepancy.

In [ ]:
simulated_background = load_events_with_truth(
    CHECKPOINT_WITHOUT_DA, H8S3_SIGNOISE + BACKGROUND_MC)
print(f"{len(simulated_background):,} simulated background events with truth")

TRUTH_VARIABLES = {
    "muon_multiplicity":  np.array([0.5, 1.5, 2.5, 3.5, 5.5, 8.5, 15.5, 30.5, 1e4]),
    "primary_energy_gev": np.logspace(2, 8, 13),
    "zenith_deg":         np.linspace(90, 180, 13),
}

required = data_tail[working_bin]
ceiling_rows = []
for variable, edges in TRUTH_VARIABLES.items():
    table = reweighting_ceiling(simulated_background, variable, edges, working_bin)
    usable = table[table["reweightable"]]
    ceiling_rows.append({
        "truth variable": variable,
        "simulation as generated": float((table["share_of_mc"]
                                          * table["pass_rate"].fillna(0)).sum()),
        "highest achievable by reweighting": float(usable["pass_rate"].max()),
        "needed to match experiment": required,
    })
ceilings = pd.DataFrame(ceiling_rows).set_index("truth variable")
ceilings["reweighting could explain it"] = (
    ceilings["highest achievable by reweighting"] >= ceilings["needed to match experiment"])
ceilings

In [ ]:
figure, axes = plt.subplots(1, len(TRUTH_VARIABLES),
                            figsize=(4.6 * len(TRUTH_VARIABLES), 4.0))
for axis, (variable, edges) in zip(axes, TRUTH_VARIABLES.items()):
    table = reweighting_ceiling(simulated_background, variable, edges, working_bin)
    axis.plot(table["bin_low"], table["pass_rate"], color=MC_COLOR, marker="o", ms=4,
              drawstyle="steps-post", label="simulation, per bin")
    axis.axhline(mc_tail[working_bin], color=MC_COLOR, ls="--", lw=1.2,
                 label="simulation, as generated")
    axis.axhline(required, color=DATA_COLOR, lw=1.6, label="experiment")
    axis.set(xlabel=variable, ylabel="fraction above the working point", yscale="log")
    if variable in ("primary_energy_gev", "muon_multiplicity"):
        axis.set_xscale("log")
    axis.grid(alpha=0.25)
axes[0].legend(fontsize=8)
figure.suptitle("Could a different background flux explain the discrepancy?", y=1.03)
figure.tight_layout()
save(figure, "d4_flux_reweighting_ceiling.png")

## 5. Is the extreme tail localised?

The ratio grows sharply towards score 1. If a few runs carried it, that would point at
data pathology rather than at modelling.

In [ ]:
TOP_BIN = int(0.99 * SCORE_BINS)
top_rows = []
for (cluster, run), frame in per_run.groupby(["cluster", "run"]):
    counts = frame.sort_values("bin")["n"].to_numpy(float)
    if counts.sum() < 5_000:
        continue
    top_rows.append({"cluster": cluster, "run": run,
                     "events passing h8s3": int(counts.sum()),
                     "events with score > 0.99": int(counts[TOP_BIN:].sum()),
                     "share with score > 0.99": fraction_above(counts, TOP_BIN)})
top_tail = pd.DataFrame(top_rows)
top_tail["experiment / simulation"] = (top_tail["share with score > 0.99"]
                                       / fraction_above(mc_counts, TOP_BIN))
top_tail.sort_values("experiment / simulation", ascending=False).reset_index(drop=True)